# ACDC Classification Report (Logs Aggregator)

This notebook aggregates cross-validation results from:
- `logs/` (all features)
- `logs_ef/` (EF-only)
- `logs_vol/` (Volumes-only, optionally calibrated)

It produces:
- A comparison table of the best model per setting (mean±std metrics)
- Per-class metrics (precision/recall/F1) aggregated across folds
- Top features (coefficients or importances) aggregated across folds
- A quick look at sample confusion matrices

> **Assumptions**: Run this notebook from the repository root so that the `logs*/` folders are in the current working directory.


In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display helper for this environment (tables appear nicely)
try:
    from caas_jupyter_tools import display_dataframe_to_user
except Exception:
    display_dataframe_to_user = None

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True, parents=True)

def read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[warn] failed to read {path}: {e}")
        return pd.DataFrame()

def load_summary(tag):
    p = Path(tag)/"cv_cls_summary.csv"
    if p.exists():
        df = pd.read_csv(p)
        df.insert(0, "setting", {"logs":"all","logs_ef":"ef","logs_vol":"vol"}.get(tag, tag))
        return df
    return pd.DataFrame()

def pick_best_per_setting(df_summary):
    if df_summary.empty:
        return pd.DataFrame()
    best = (df_summary.sort_values(["setting","acc_mean"], ascending=[True,False])
                      .groupby("setting").head(1).reset_index(drop=True))
    return best

def show_df(df, name):
    if display_dataframe_to_user:
        display_dataframe_to_user(name, df)
    else:
        display(df.head())

print("Notebook ready. Make sure 'logs/', 'logs_ef/', 'logs_vol/' exist in CWD.")

## 1) Best Model per Setting

In [ ]:
tags = ["logs","logs_ef","logs_vol"]
frames = [load_summary(t) for t in tags if (Path(t)/"cv_cls_summary.csv").exists()]
agg_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
best = pick_best_per_setting(agg_all)

print("=== Best model per setting ===")
cols = ["setting","model","acc_mean","acc_std","f1m_mean","f1m_std","aucm_mean","aucm_std"]
best_disp = best[cols] if not best.empty else pd.DataFrame()
show_df(best_disp, "Best model per setting")

# Save CSVs
agg_all.to_csv(reports_dir/"cls_aggregate_all.csv", index=False)
best_disp.to_csv(reports_dir/"cls_aggregate_best.csv", index=False)
print("Saved:", reports_dir/"cls_aggregate_all.csv")
print("Saved:", reports_dir/"cls_aggregate_best.csv")

## 2) Per-Class Metrics (Aggregated across folds)

In [ ]:
def perclass_from_reports(tag, model_name):
    """Collect per-class precision/recall/F1 across folds and compute mean±std."""
    files = sorted(glob.glob(f"{tag}/cv_cls_{model_name}_fold*_perclass.csv"))
    if not files:
        return pd.DataFrame()
    dfs = []
    for f in files:
        df = read_csv_safe(f)
        # Keep only disease classes (rows) if present
        keep = [c for c in ["DCM","HCM","MINF","NOR","RV"] if c in df.index]
        if not keep and "support" in df.columns:
            # Sometimes the index may be a column
            if "Unnamed: 0" in df.columns:
                df = df.set_index("Unnamed: 0")
                keep = [c for c in ["DCM","HCM","MINF","NOR","RV"] if c in df.index]
        df = df.loc[keep] if keep else df
        df["__fold__src"] = f
        dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    big = pd.concat(dfs, axis=0)
    # Aggregate mean±std per class
    out = big.groupby(big.index).agg({
        "precision": ["mean","std"],
        "recall": ["mean","std"],
        "f1-score": ["mean","std"],
        "support": "sum"
    })
    # Flatten columns
    out.columns = ["_".join([a,b]) if isinstance(b,str) else a for a,b in out.columns]
    return out

perclass_tables = {}
if not best.empty:
    for _, row in best.iterrows():
        tag = {"all":"logs","ef":"logs_ef","vol":"logs_vol"}[row["setting"]]
        model = row["model"]
        print(f"Aggregating per-class metrics for setting={row['setting']} model={model}")
        tbl = perclass_from_reports(tag, model)
        perclass_tables[(row["setting"], model)] = tbl
        # Save and show
        out_csv = reports_dir / f"perclass_{row['setting']}_{model}.csv"
        tbl.to_csv(out_csv)
        print("Saved:", out_csv)
        show_df(tbl.reset_index(), f"Per-class ({row['setting']} | {model})")
else:
    print("[warn] No 'best' rows found (missing logs?).")

## 3) Plots: Per-Class F1 (mean across folds)

In [ ]:
for (setting, model), tbl in perclass_tables.items():
    if "f1-score_mean" not in tbl.columns:
        continue
    plt.figure(figsize=(6,4))
    tbl["f1-score_mean"].plot(kind="bar")
    plt.title(f"Per-class F1 (mean) — {setting} | {model}")
    plt.ylabel("F1")
    plt.tight_layout()
    fig_path = reports_dir / f"plot_perclass_f1_{setting}_{model}.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print("Saved:", fig_path)

## 4) Feature Importance / Coefficients (Aggregated across folds)

In [ ]:
def load_feature_importance(tag, model):
    # Prefer tree importances if present
    files = sorted(glob.glob(f"{tag}/feature_importance/fi_{model}_fold*.csv"))
    if files:
        series_list = []
        for f in files:
            s = read_csv_safe(f)
            if s.shape[1]==1:  # read as single column -> fix
                s = s.squeeze()
            else:
                # If read_csv returns 2 columns, try to set first as index
                try:
                    s = pd.Series(s.iloc[:,1].values, index=s.iloc[:,0].values)
                except Exception:
                    s = pd.Series(dtype=float)
            series_list.append(s)
        if series_list:
            df = pd.concat(series_list, axis=1)
            df.columns = [Path(x).stem for x in files]
            mean_imp = df.mean(axis=1).sort_values(ascending=False)
            return mean_imp
    # Else try LR coefficients: average absolute coefficients across classes and folds
    files = sorted(glob.glob(f"{tag}/feature_importance/coef_{model}_fold*.csv"))
    if files:
        per_fold = []
        for f in files:
            try:
                df = pd.read_csv(f, index_col=0)
            except Exception:
                df = read_csv_safe(f)
            if df.empty:
                continue
            # abs across classes then mean to get per-feature strength
            abs_mean = df.abs().mean(axis=0)
            per_fold.append(abs_mean)
        if per_fold:
            coef = pd.concat(per_fold, axis=1).mean(axis=1).sort_values(ascending=False)
            return coef
    return pd.Series(dtype=float)

for _, row in best.iterrows():
    tag = {"all":"logs","ef":"logs_ef","vol":"logs_vol"}[row["setting"]]
    model = row["model"]
    imp = load_feature_importance(tag, model)
    if imp.empty:
        print(f"[warn] No importances/coefficients found for {row['setting']} | {model}.")
        continue
    topk = imp.head(10)
    plt.figure(figsize=(6,4))
    topk.plot(kind="barh")
    plt.title(f"Top features — {row['setting']} | {model}")
    plt.xlabel("Importance / |coef| (mean)")
    plt.tight_layout()
    fig_path = reports_dir / f"plot_top_features_{row['setting']}_{model}.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print("Saved:", fig_path)

## 5) Sample Confusion Matrices

In [ ]:
def show_some_cms(tag, model, max_imgs=3):
    files = sorted(glob.glob(f"{tag}/cv_cls_{model}_fold*_cm.png"))
    for i, f in enumerate(files[:max_imgs]):
        img = plt.imread(f)
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(Path(f).name)
        plt.tight_layout()
        plt.show()

if not best.empty:
    for _, row in best.iterrows():
        tag = {"all":"logs","ef":"logs_ef","vol":"logs_vol"}[row["setting"]]
        model = row["model"]
        print(f"Showing confusion matrices for {row['setting']} | {model}:")
        show_some_cms(tag, model, max_imgs=3)
else:
    print("[warn] No best rows found, skipping CMs.")